## **Import libraries** for data analysis, JSON handling, progress tracking, and file download.
### `pandas` is used for tabular data.
### `json` is used because the API will return structured JSON output.
### `tqdm` helps show progress when processing multiple rows.
### `gdown` downloads the dataset from Google Drive.

In [2]:
import pandas as pd
import json
import tqdm
import gdown

### Download the survey dataset from Google Drive and load it into a DataFrame.


In [3]:
file_id = "1J424k-xkNvq349TyrD9m2s_iDywrHeds"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "workshop_survey_syn_dataset.csv",
    quiet=False
)

data = pd.read_csv(r'workshop_survey_syn_dataset.csv')

Downloading...
From: https://drive.google.com/uc?id=1J424k-xkNvq349TyrD9m2s_iDywrHeds
To: /content/workshop_survey_syn_dataset.csv
100%|██████████| 440k/440k [00:00<00:00, 6.58MB/s]


### Data quality check

Before using AI, we inspect the dataset so we know what data is being analyzed. This keeps the workflow transparent and helps avoid treating AI output as a black box.

### Check the `size` of the dataset: number of rows and columns.

In [4]:
data.shape

(1000, 9)

### Check the `data type` of each column.


In [5]:
data.dtypes

,0
participant_id,int64
role,object
department,object
comfort_with_ai,object
main_interest_category,object
main_interest_detail,object
concern_category,object
concern_detail,object
free_text_feedback,object


### Display all column names available in the dataset.

In [6]:
data.columns

Index(['participant_id', 'role', 'department', 'comfort_with_ai',
       'main_interest_category', 'main_interest_detail', 'concern_category',
       'concern_detail', 'free_text_feedback'],
      dtype='object')

### Preview the first 10 rows.

This is to understand the structure of the participant responses before sending anything to the AI.

In [7]:
data.head(10)

,participant_id,role,department,comfort_with_ai,main_interest_category,main_interest_detail,concern_category,concern_detail,free_text_feedback
0,837645,Faculty,Public Health,Comfortable,Grant Writing,Improving clarity and turnaround on competitiv...,Hallucinations and Accuracy,Hard to verify AI output accuracy for technica...,"I already use AI fairly regularly, I'd especia..."
1,475133,Undergraduate Student,Education,Neutral,Student Learning Support,Helping Education students use AI tutors witho...,Academic Misconduct,Students using AI to complete Education assign...,"I have mixed feelings about AI, I really want ..."
2,601099,Faculty,Biology,Uncomfortable,Teaching and Course Design,Designing AI-aware assignments and rubrics for...,Copyright and Intellectual Property,Unclear copyright and ownership rules for AI o...,"I've only dabbled with AI and feel uncertain, ..."
3,269917,Research Scientist,Computer Science,Comfortable,Data Analysis,"Cleaning, exploring, and visualizing Computer ...",Data Privacy,Unclear where AI vendors store and reuse our C...,"I already use AI fairly regularly, I'm keen to..."
4,698457,Administrative Staff,Medicine,Neutral,Research Productivity,Cutting time spent on repetitive analysis step...,Data Privacy,Protecting confidential participant and studen...,"I have mixed feelings about AI, I'd love pract..."
5,556706,Undergraduate Student,Medicine,Comfortable,Programming Assistance,Getting help writing and debugging code for my...,No Major Concerns,Generally optimistic about AI's role in suppor...,I'm reasonably comfortable using AI day to day...
6,467320,Administrative Staff,Information Technology,Uncomfortable,Research Compliance,Ensuring data handling and AI use meet Informa...,No Major Concerns,Generally optimistic about AI's role in suppor...,"I've only dabbled with AI and feel uncertain, ..."
7,255499,Undergraduate Student,Public Health,Comfortable,Career Development,Building AI skills that strengthen my Public H...,Student Overreliance on AI,Worried Public Health students stop thinking c...,I'm reasonably comfortable using AI day to day...
8,660112,Research Scientist,Research Computing,Comfortable,Research Productivity,Cutting time spent on repetitive analysis step...,Student Overreliance on AI,Overreliance eroding foundational competencies...,I'm reasonably comfortable using AI day to day...
9,158089,Graduate Student,Biology,Very Comfortable,Programming Assistance,Getting help writing and debugging code for my...,Research Integrity,Worried AI-generated text could undermine inte...,"I'm an enthusiastic adopter of AI tools, I'm h..."


### Using AI Externally

In this section, we use an AI model through an API instead of manually uploading the dataset into ChatGPT or Claude.

The goal is to make the AI step more systematic, repeatable, and easier to apply across many rows.

In [8]:
from google.colab import userdata

API_KEY = userdata.get('MY_API_KEY')

# Add your API key here.
# In a real workflow, avoid hardcoding API keys directly in notebooks.
# Use environment variables or secure notebook secrets when possible.

In [9]:
# Import the OpenAI client.
from openai import OpenAI

In [10]:
# Create the API client using the provided API key.
# This client is what the notebook uses to send prompts to the AI model.

client = OpenAI(api_key=API_KEY)

### RQ1: Who participated, and how AI-comfortable are different groups?


In [11]:
role_counts = data["role"].value_counts()
department_counts = data["department"].value_counts()
comfort_counts = data["comfort_with_ai"].value_counts()

role_by_comfort_counts = pd.crosstab(
    data["role"],
    data["comfort_with_ai"]
)

role_by_comfort_percent = pd.crosstab(
    data["role"],
    data["comfort_with_ai"],
    normalize="index"
).round(2)

role_counts, department_counts, comfort_counts, role_by_comfort_counts, role_by_comfort_percent

(role
 Undergraduate Student      251
 Graduate Student           197
 Faculty                    153
 Research Staff              78
 Administrative Staff        75
 Research Scientist          56
 Postdoctoral Researcher     51
 Academic Advisor            42
 IT Staff                    36
 Department Chair            23
 Academic Leadership         21
 Librarian                   17
 Name: count, dtype: int64,
 department
 Public Health             129
 Medicine                  119
 Computer Science          112
 Engineering               105
 Biology                    94
 Business                   92
 Humanities                 88
 Education                  83
 Research Computing         43
 Student Affairs            41
 Sponsored Programs         38
 Information Technology     33
 Library Services           23
 Name: count, dtype: int64,
 comfort_with_ai
 Comfortable           338
 Neutral               283
 Very Comfortable      204
 Uncomfortable         134
 Very Uncomfor

In [ ]:
# The AI is not calculating the counts; pandas already did it.
# Here AI is helping translate the tables into an user-friendly interpretation.

RQ1_prompt = f"""
You are helping explain workshop survey results to a non-technical audience.

Research Question 1:
Who participated in the survey, and how AI-comfortable are different groups?

Use the summary tables below to write a clear answer.

Role counts:
{role_counts.to_string()}

Department counts:
{department_counts.to_string()}

Overall AI comfort counts:
{comfort_counts.to_string()}

Role by AI comfort counts:
{role_by_comfort_counts.to_string()}

Role by AI comfort percentages:
{role_by_comfort_percent.to_string()}

Write:
1. A concise answer to the research question
2. Two or three important patterns
3. One sentence explaining why this matters for planning future AI support
"""

RQ1_response = client.responses.create(
    model="gpt-4.1-mini",
    input=RQ1_prompt
)

print(RQ1_response.output_text)

1. **Answer to the research question:**
The survey included a diverse group of 1,059 participants across roles such as undergraduate and graduate students, faculty, research staff, and administrative personnel, with representation from many departments including Public Health, Medicine, Computer Science, and more. Overall, about half of respondents felt comfortable or very comfortable with AI, though comfort levels varied by role, with graduate students and research scientists showing higher AI comfort compared to faculty and administrative staff.

2. **Important patterns:**
- Graduate students and research scientists reported higher levels of AI comfort (around 39-38% comfortable and 20-21% very comfortable) compared to faculty, who had comparatively lower comfort levels (29% comfortable and 16% very comfortable) and a higher proportion feeling uncomfortable.
- Administrative roles and faculty had the highest percentages of respondents who felt uncomfortable or very uncomfortable with

### RQ2: What do participants want to use AI for, and what concerns are attached to those interests?


In [12]:
# RQ2: What do participants want to use AI for, and what concerns are attached to those interests?

interest_counts = data["main_interest_category"].value_counts()
concern_counts = data["concern_category"].value_counts()

role_by_interest = pd.crosstab(
    data["role"],
    data["main_interest_category"]
)

interest_by_concern_counts = pd.crosstab(
    data["main_interest_category"],
    data["concern_category"]
)

interest_by_concern_percent = pd.crosstab(
    data["main_interest_category"],
    data["concern_category"],
    normalize="index"
).round(2)

interest_counts, concern_counts, role_by_interest, interest_by_concern_counts, interest_by_concern_percent

(main_interest_category
 Student Learning Support      105
 Programming Assistance        100
 Generative AI Tools            99
 Academic Writing               97
 Grant Writing                  79
 Literature Review              76
 Research Productivity          73
 Career Development             71
 Data Analysis                  68
 Administrative Efficiency      57
 Responsible AI                 43
 Research Compliance            36
 Teaching and Course Design     35
 AI Policy and Governance       33
 Research Computing             28
 Name: count, dtype: int64,
 concern_category
 Data Privacy                           130
 Academic Misconduct                    118
 Hallucinations and Accuracy            116
 Research Integrity                     115
 Lack of Training                       103
 Student Overreliance on AI             103
 No Major Concerns                       90
 Copyright and Intellectual Property     68
 Security Risks                          42
 Cost and

In [ ]:
RQ2_prompt = f"""
You are helping explain workshop survey results to a non-technical audience.

Research Question 2:
What do participants want to use AI for, and what concerns are attached to those interests?

Use the summary tables below to write a clear answer.

Main AI interest counts:
{interest_counts.to_string()}

Concern category counts:
{concern_counts.to_string()}

Role by main interest:
{role_by_interest.to_string()}

Main interest by concern counts:
{interest_by_concern_counts.to_string()}

Main interest by concern percentages:
{interest_by_concern_percent.to_string()}

Write:
1. A concise answer to the research question
2. The most common AI use cases
3. The most common concerns
4. Any important relationship between interests and concerns
5. What this means for designing future AI workshops
"""

RQ2_response = client.responses.create(
    model="gpt-4.1-mini",
    input=RQ2_prompt
)

print(RQ2_response.output_text)

1. **Answer to Research Question 2:**  
Participants primarily want to use AI to support student learning, assist with programming, and aid academic and grant writing. However, these interests come with significant concerns, especially around data privacy, potential academic misconduct, and the accuracy of AI outputs. There is also worry about students over-relying on AI and a general lack of training to use AI effectively and responsibly.

2. **Most Common AI Use Cases:**  
- Student Learning Support  
- Programming Assistance  
- Generative AI Tools  
- Academic Writing  
- Grant Writing  

3. **Most Common Concerns:**  
- Data Privacy  
- Academic Misconduct  
- Hallucinations (incorrect AI outputs) and Accuracy  
- Research Integrity  
- Lack of Training  
- Student Overreliance on AI  

4. **Important Relationships Between Interests and Concerns:**  
- Those interested in **Student Learning Support** and **Academic Writing** expressed high concerns about academic misconduct, data 

# RQ3

In [ ]:
# Define the allowed support pathways.
'''
These categories limit the AI output so it chooses
from this list instead of inventing new ones.
'''
support_pathways = [
    "AI Technical Consulting",
    "Teaching AI Literacy",
    "Architecture Development",
    "Infrastructure Support",
    "Research workflow consultation",
    "Teaching and Course Design Support",
    "Grant Writing Support",
    "Hackathons or workshop support",
    "Responsible AI establishment"
]

In [ ]:
# Define a JSON schema for the AI response.
# This forces the model to return structured output with the same fields every time we execute it.

schema_support_pathways = {
    "type": "object",
    "properties": {
        "Recommended_support_pathway": {
            "type": "string",
            "enum": support_pathways
        },
        "Brief_reason": {
            "type": "string"
        },
        "Explaination_phrase": {
            "type": "string"
        }
    },
    "required": ["Recommended_support_pathway", "Brief_reason", "Explanation_phrase"],
    "additionalProperties": False
}

In [ ]:
# Build the prompt for one participant row.
# The prompt gives the AI structured survey information and asks it to choose
# the best support pathway.

def prompt_function(row):
  return f"""
  You are analyzing a workshop survey feedback.

  The task here is to classify the participant into one recommend support pathway.

  Available support pathways:
  {chr(10).join("- " + p for p in support_pathways)}

  Participant information:
  Role: {row["role"]}
  Department: {row["department"]}
  Comfort with AI: {row["comfort_with_ai"]}
  Main interest: {row["main_interest_category"]} - {row["main_interest_detail"]}
  Concern: {row["concern_category"]} - {row["concern_detail"]}
  Free-text feedback: {row["free_text_feedback"]}

  Use all of the available information to make a recommendation.

  Choose the single pathway that would provide the greatest immediate benefit.
  If multiple pathways apply, select the one of themost directly supported by the participant's stated interest and concern.

  Return:
  - Recommended_support_pathway
  - Brief_reason
  - Explanation_phrase
  """

In [ ]:
# Send one participant's prompt to the AI model.
# The model returns a structured JSON response following the schema defined above.

def support_pathway_addition(row):
  prompt = prompt_function(row)

  response = client.responses.create(
      model="gpt-4.1-mini",
      input=prompt,
      text={
          "format": {
              "type": "json_schema",
              "name": "support_pathway_schema",
              "schema": schema_support_pathways,
              "strict": True
          }
      }
  )

  return json.loads(response.output_text)

###Error Handling

When using an API, some requests may fail because of `formatting issues`, `network problems`, `rate limits`, or `invalid responses`. This following function catches errors so the workflow can continue instead of stopping completely.

In [ ]:
# Classify one row and safely handle errors.
# If the API call works, we return the recommended pathway, reason, and evidence phrase.
# If it fails, we return empty values and save the error message.

def classify_row(row):
  try:
    result = support_pathway_addition(row)

    return {
        "Recommended_support_pathway": result['Recommended_support_pathway'],
      "Brief_reason": result['Brief_reason'],
      "Explanation_phrase": result['Explanation_phrase']
    }
  except Exception as e:
    return {
        "Recommended_support_pathway": None,
        "Brief_reason": None,
        "Explaination_phrase": None,
        "status": str(e)
    }

### Test the workflow on one row

Before running an AI workflow on many rows, we test it on one example. This helps confirm that the `prompt`, `schema`, and `API connection` are working.

In [ ]:
test_one_row = classify_row(data.iloc[0])
print(test_one_row)

{'Recommended_support_pathway': 'Grant Writing Support', 'Brief_reason': 'Participant is a faculty member in Public Health who is comfortable with AI and primarily interested in improving grant writing clarity and turnaround.', 'Explaination_phrase': "Given the participant's role and their expressed main interest in improving Public Health grant writing, along with concerns about AI accuracy, providing targeted support for grant writing will directly address their immediate needs and maximize benefit."}


### Complete dataset

In [ ]:
# Run the classification workflow on a selected range of rows.
# The progress bar helps us track how many rows have been processed.
# You can customize the no of rows from run_partition(data, start, end) within
# the size of the dataset

def run_partition(data, start, end):
    subset = data.iloc[start:end]
    results_set = []

    for _, row in tqdm.tqdm(
        subset.iterrows(),
        total=len(subset),
        desc=f"{start}:{end}"
    ):
        results_set.append(classify_row(row))

    return results_set

results_set = run_partition(data, 0, 100)

0:100: 100%|██████████| 100/100 [00:00<00:00, 10379.63it/s]


In [ ]:
# Convert the API results into a DataFrame and attach them to the original survey data.
# Creating a dataset with both original survey fields and AI-generated analysis.

dataset = pd.DataFrame(results_set)

analyzed_data = pd.concat([
    data.reset_index(drop=True),
    dataset.reset_index(drop=True)],
                          axis=1
                          )

In [ ]:
analyzed_data.head()

,participant_id,role,department,comfort_with_ai,main_interest_category,main_interest_detail,concern_category,concern_detail,free_text_feedback,Recommended_support_pathway,Brief_reason,Explaination_phrase
0,837645,Faculty,Public Health,Comfortable,Grant Writing,Improving clarity and turnaround on competitiv...,Hallucinations and Accuracy,Hard to verify AI output accuracy for technica...,"I already use AI fairly regularly, I'd especia...",Grant Writing Support,The participant explicitly expressed interest ...,Given the participant's focus on enhancing Pub...
1,475133,Undergraduate Student,Education,Neutral,Student Learning Support,Helping Education students use AI tutors witho...,Academic Misconduct,Students using AI to complete Education assign...,"I have mixed feelings about AI, I really want ...",Teaching AI Literacy,The participant is focused on supporting stude...,Given the participant's role as an Education u...
2,601099,Faculty,Biology,Uncomfortable,Teaching and Course Design,Designing AI-aware assignments and rubrics for...,Copyright and Intellectual Property,Unclear copyright and ownership rules for AI o...,"I've only dabbled with AI and feel uncertain, ...",Teaching and Course Design Support,The participant is a faculty member in Biology...,Given the participant's focus on teaching and ...
3,269917,Research Scientist,Computer Science,Comfortable,Data Analysis,"Cleaning, exploring, and visualizing Computer ...",Data Privacy,Unclear where AI vendors store and reuse our C...,"I already use AI fairly regularly, I'm keen to...",Responsible AI establishment,The participant is comfortable with AI and act...,Given the participant's comfort with AI and fo...
4,698457,Administrative Staff,Medicine,Neutral,Research Productivity,Cutting time spent on repetitive analysis step...,Data Privacy,Protecting confidential participant and studen...,"I have mixed feelings about AI, I'd love pract...",Research workflow consultation,The participant seeks practical ways to enhanc...,Given the participant's focus on improving res...
